# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
!git clone https://github.com/Prithviraj-chw/FlyRank-Starter.git
%cd FlyRank-Starter
!ls data/raw/

Cloning into 'FlyRank-Starter'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 98 (delta 20), reused 79 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 1.83 MiB | 7.93 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/FlyRank-Starter/FlyRank-Starter/FlyRank-Starter
content_refresh_anonymized.csv


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Lane: Refresh / Content Opportunity Scoring.

I'm picking this over the other three lanes because it's a scoring-and-ranking
problem — take a set of items, score them, output a prioritized action list —
which is the same shape as segmentation/prioritization work I've already built
(RFM + K-Means customer segmentation with a promotion-recommendation layer).
It also gives me a clear baseline-vs-model comparison to test, rather than
open-ended EDA (Lane 1) or an unlabeled clustering problem (Lane 3).

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

1. What decision does this improve?
   Which content pages a reviewer should look at first, out of a much
   larger inventory, given limited review time.

2. Who acts on the output, and what do they do?
   A content reviewer with fixed weekly capacity. They open the ranked
   queue top-down and take one of: refresh, expand, protect, prune, or
   monitor — based on the reason code attached to each page.

3. What does a wrong answer cost?
   False positive: a reviewer spends limited time on a page that didn't
   need it. False negative: a genuinely declining, high-demand page never
   surfaces, and keeps losing visibility unnoticed. Since false negatives
   are more expensive here, I care about recall as well as precision@K,
   not precision alone.

4. Why does data or ML help at all?
   A single hand-written rule (e.g. "flag if stale AND high-impressions")
   can't weigh several weak, tangled signals together the way a model can.
   The starter pipeline already shows this isn't theoretical: the fixed
   rule hit precision@50 = 0.240, a random forest on the same signals hit
   0.740 — roughly 12 vs. 37 correct picks in the top 50.

One-paragraph frame:
For a content reviewer with limited weekly capacity, deciding which pages
to review first, we will build a ranked opportunity queue from observable
search and engagement signals, scoring priority for refresh/expand/protect/
prune/monitor, measured by precision@50 and recall against a defined
decline/opportunity outcome. A wrong call costs either wasted reviewer time
or a missed real decline. A plain rule isn't enough because it can't
combine multiple weak, shifting signals the way a model can. We will claim
only observed, directional, decision-support results — never that a refresh
will cause a recovery.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [10]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df.head()


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [11]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

In [12]:
filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
filtered = filtered.drop_duplicates(subset="content_id")

decline_pct = (filtered["trend_direction"] == "down").mean() * 100
stale_visible = filtered[(filtered["days_since_last_update"] >= 180) & (filtered["impressions_90d"] >= 500)]

print(f"Rows after filtering: {len(filtered)}")
print(f"% currently trend_direction == 'down': {decline_pct:.1f}%")
print(f"Pages matching 'stale_visible_page' rule: {len(stale_visible)} ({len(stale_visible)/len(filtered)*100:.1f}%)")
print(f"Median impressions_90d: {filtered['impressions_90d'].median()}")

Rows after filtering: 30000
% currently trend_direction == 'down': 54.2%
Pages matching 'stale_visible_page' rule: 17 (0.1%)
Median impressions_90d: 731.0


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

This work can say: which pages show observed signs of decline or opportunity
right now, and how a learned ranking compares to a fixed rule on this data.
It is directional and decision-support only — a prioritized list for a human
to review, not a guarantee.

This work cannot say: that a refresh will cause a recovery (that needs an
experiment), that any result reveals a Google ranking factor, or that a
score is "the truth" rather than one model's estimate.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.